In [ ]:
%%time
from ardal import Ardal
import pandas as pd
import time
import numpy as np
from scipy.stats import binom
from collections import namedtuple

# data = "/home/arthur/BioInf/Crypto_popgen/data/concat_crypto.csv"
# metadata_file = "/home/arthur/BioInf/Crypto_popgen/data/crypto_cats_with_het.csv"
# meta_df = pd.read_csv(metadata_file, index_col=0)

data = ["/home/arthur/BioInf/SSD/BioInf/Plasmodium_vcfs/Pf_matrix.npy", "/home/arthur/BioInf/SSD/BioInf/Plasmodium_vcfs/Pf_headers.json"]
metadata_file = "/home/arthur/BioInf/ProtoDB/WD/pf/Pf7_meta.csv"
meta_df = pd.read_csv(metadata_file, index_col=0)

ard = Ardal(data, force_roaring=True)

In [ ]:
def get_discrim_snps(ingroup_guids, ard):
    snps = ard.snpInform(ingroup_guids, metric="informationgain")
    percentile = np.percentile(list(snps.values()), 99.999)
    print(f"Percentile: {percentile}")
    selected_snps = [snp for snp, kl in snps.items() if kl >= percentile]

    print(f"Selected {len(selected_snps)} SNPs with KL divergence >= {percentile}")

    cooc = ard.alleleCooc(selected_snps, threshold=0.95, threads=10)
    na_snps = [snp for d in list(cooc.values()) for snp in d]
    
    print(f"Filtered {len(na_snps)} SNPs after co-occurrence filtering")
    selected_snps = [snp for snp in selected_snps if snp not in na_snps]
    print(f"Final selected SNPs: {len(selected_snps)}\n\n")
    return selected_snps


def freqs(lineage_matrix):
    alpha = 1
    lin_freqs = []
    for lin in lineage_matrix:
        f = (np.sum(lin, axis=0) + alpha) / (lin.shape[0] + 2 * alpha)
        lin_freqs.append(f)
        
    return lin_freqs


def compute_likelihoods(q, af_distributions):
    log_likelihoods = []
    for af_i, af in enumerate(af_distributions):
        l = 0
        for i, b in enumerate(q):
            if b == 1:
                l += np.log(af[i])
                # l += w[i] * np.log(af[i])
            else:
                l += np.log(1-af[i])
                # l += w[i] * np.log(1-af[i])
        log_likelihoods.append(l)
    return log_likelihoods


def compute_surprise(q, af_distribution):
    K, d = af_distributions.shape
    anoms = np.zeros((K, d))
    for k in range(K):
        for i, b in enumerate(q):
            if b == 1:
                anoms[k, i] = -np.log(af_distributions[k, i])
            else:
                anoms[k, i] = -np.log(1 - af_distributions[k, i])
    return anoms


def compute_binomial_loglik_binary(q, af_distribution):
    trials = zip(q, af_distribution)
    
    logliks = 0
    for x, p in trials:
        logliks += binom.logpmf(x, 1, p)
    
    return logliks


def detect_anomalies(anoms, tau=1):
    Anomaly = namedtuple('Anomaly', ['score', 'pos_tuple', 'surprise'])
    anomalies = []
    for arr in anoms:
        agg_anomaly_score = np.sum(arr[arr>=tau])
        arr_an = Anomaly(agg_anomaly_score, [i for i, x in enumerate(arr) if x > tau], [x for i, x in enumerate(arr) if x > tau])
        anomalies.append(arr_an)
    return anomalies

# def compute_bayes(log_likelihoods):
#     priors = 1/len(log_likelihoods)
#     esum_ls = np.sum([np.exp(l) for l in log_likelihoods])
#     probs = []
#     for l in log_likelihoods:
#         p = np.exp(l)/esum_ls
#         probs.append(p)
#     return probs

def compute_bayes(log_likelihoods):
    esum_ls = np.sum([np.exp(l) for l in log_likelihoods])
    probs = [np.exp(l)/esum_ls for l in log_likelihoods]
    return probs

In [ ]:
ingroup_str = """SRR21763624_Calf10
SRR21763626_Calf8
SRR21763629_Lamb5
SRR21763631_Lamb3
SRR21763633_Calf13
SRR21763636_Lamb1
SRR26320644_UKP125
SRR26320645_UKP124
SRR26320655_UKP104
SRR26320657_UKP102
SRR26320663_Swe2
SRR26320681_Hun2
SRR26320696_GER21
SRR26320709_FIN2
SRR26320733_FIN4
SRR26320737_FIN1
SRR10363424_41567
"""
ingroup = [x.strip() for x in ingroup_str.split("\n") if x.strip()]

ingroup = [i for i in meta_df[meta_df["continent"] == "Europe"].index.tolist() if i in ard.getHeaders()["guids"]]

In [ ]:
meta_df = pd.read_csv(metadata_file)
meta = meta_df[meta_df['guid'].isin(ard.getHeaders()['guids'])]
print(meta['subtype'].value_counts())

In [ ]:
pop="AS-SE-E"
print(meta['subtype'].value_counts())
ingroup_guids = list([i for i in meta[meta['subtype']==pop]['guid'] if i in ard.getHeaders()['guids']])
len(ingroup_guids)

discrim_snps = get_discrim_snps(ingroup_guids, ard)

In [ ]:
%%time
kl = ard.snpInform(ingroup_guids, metric="kullbackleibler")
js = ard.snpInform(ingroup_guids, metric="jensenshannon")
ig = ard.snpInform(ingroup_guids, metric="informationgain")

In [ ]:
d = {}
for k, v in kl.items():
    d[k] = {"kl": v, "js": js[k], "ig": ig[k]}
df = pd.DataFrame(d).T
df

In [15]:
pop_models = {}

for pop in set(meta['subtype'].values):
    if pop == "unknown":
        continue

    ingroup_guids = list([i for i in meta[meta['subtype']==pop]['guid'] if i in ard.getHeaders()['guids']])
    if len(ingroup_guids) < 10:
        print(f"Skipping population {pop} with only {len(ingroup_guids)} ingroup GUIDs.")
        continue
    
    print(f"Processing population: {pop} with {len(ingroup_guids)} ingroup GUIDs.")
    
    discrim_snps = get_discrim_snps(ingroup_guids, ard)

    ingroup_mat, ingroup_headers = ard.subset(guid_list=ingroup_guids, allele_list=discrim_snps)
    f = freqs(ingroup_mat)
    pop_models[pop] = list(zip(discrim_snps, f))

Skipping population SA with only 1 ingroup GUIDs.
Processing population: AF-NE with 42 ingroup GUIDs.
Percentile: 0.02202466023249949
Selected 75 SNPs with KL divergence >= 0.02202466023249949
Filtered 14 SNPs after co-occurrence filtering
Final selected SNPs: 67


Processing population: AS-SE-W with 196 ingroup GUIDs.
Percentile: 0.1391363016476479
Selected 76 SNPs with KL divergence >= 0.1391363016476479
Filtered 63 SNPs after co-occurrence filtering
Final selected SNPs: 59


Processing population: AS-S-FE with 170 ingroup GUIDs.
Percentile: 0.10813045964440626
Selected 75 SNPs with KL divergence >= 0.10813045964440626
Filtered 53 SNPs after co-occurrence filtering
Final selected SNPs: 56


Processing population: AF-E with 555 ingroup GUIDs.
Percentile: 0.18189399046775745
Selected 75 SNPs with KL divergence >= 0.18189399046775745
Filtered 3 SNPs after co-occurrence filtering
Final selected SNPs: 73


Processing population: AF-W with 1236 ingroup GUIDs.
Percentile: 0.3400778236212707

In [16]:
pop_models

{'AF-NE': [('Pf3D7_06_v3.1147607.A', np.float64(0.5072463768115942)),
  ('Pf3D7_06_v3.1124712.A', np.float64(0.9130434782608695)),
  ('Pf3D7_API_v3.4030.C', np.float64(0.9130434782608695)),
  ('Pf3D7_08_v3.942364.T', np.float64(0.8260869565217391)),
  ('Pf3D7_06_v3.1168901.G', np.float64(0.7971014492753623)),
  ('Pf3D7_06_v3.1149900.G', np.float64(0.7971014492753623)),
  ('Pf3D7_06_v3.1145524.G', np.float64(0.2753623188405797)),
  ('Pf3D7_06_v3.1148402.A', np.float64(0.782608695652174)),
  ('Pf3D7_06_v3.1154545.C', np.float64(0.8260869565217391)),
  ('Pf3D7_06_v3.1131659.T', np.float64(0.8115942028985508)),
  ('Pf3D7_06_v3.1112523.T', np.float64(0.7681159420289855)),
  ('Pf3D7_06_v3.1165135.C', np.float64(0.5942028985507246)),
  ('Pf3D7_09_v3.578034.C', np.float64(0.7246376811594203)),
  ('Pf3D7_06_v3.1201306.A', np.float64(0.7246376811594203)),
  ('Pf3D7_06_v3.1203867.A', np.float64(0.6086956521739131)),
  ('Pf3D7_06_v3.1175227.A', np.float64(0.7101449275362319)),
  ('Pf3D7_06_v3.1132

In [ ]:
outgroup_guids = [g for g in ard.getHeaders()["guids"] if g not in ingroup_guids]
outgroup_afs = ard.af(outgroup_guids)
selected_snps_oafs = [[i, outgroup_afs[i]] for i in discrim_snps]
selected_snps_oafs

In [ ]:
# mat, headers = ard.subset(allele_list=selected_snps)
# ingroup_mat, ingroup_headers = ard.subset(guid_list=ingroup_guids, allele_list=selected_snps)
# outgroup_mat, outgroup_headers = ard.subset(guid_list=outgroup_guids, allele_list=selected_snps)

pops = list(mats.keys())
matrices = list(mats.values())
af_distributions = freqs(matrices)

print(af_distributions)

In [ ]:
for i in range(len(af_distributions)):
    print(len(af_distributions[i]), len(list(weights.values())[i]))
    print(af_distributions[i]*list(weights.values())[i])

In [ ]:
idx = []

t=0.95

results = {"TP" : 0, "FP" : 0, "TN" : 0, "FN" : 0}
r2 = {}

for i, q in enumerate(mat):
    guid = headers['guids'][i]
    guid_pop = meta[meta['guid']==guid]['subtype'].values[0]

    log_likelihoods = compute_likelihoods(q, af_distributions, list(weights.values()))
    # print(f"Log likelihoods for {headers['guids'][i]}: {log_likelihoods}")
    probs = compute_bayes(log_likelihoods)

    r2[guid] = {}
    r2[guid]["pop"] = guid_pop

    pop_probs = list(zip(pops, probs))

    for pop_model, prob in pop_probs:
        r2[guid][pop_model] = prob

    
    # if guid in ingroup_guids and probs[0]>=t:
    #     results["TP"] += 1

    # if guid in ingroup_guids and probs[0]<t:
    #     results["FN"] += 1

    # if guid in outgroup_guids and probs[0]>=t:
    #     results["FP"] += 1
    
    # if guid in outgroup_guids and probs[0]<t:
    #     results["TN"] += 1

df = pd.DataFrame(r2).T
print(results)
df

In [ ]:
from collections import defaultdict

t=0.95

results = {"TP" : 0, "FP" : 0, "TN" : 0, "FN" : 0}
r2 = defaultdict(dict)

for pop, model in pop_models.items():
    print(f"Running model: {pop}")

    ingroup_guids = list([i for i in meta[meta['subtype']==pop]['guid'] if i in ard.getHeaders()['guids']])
    outgroup_guids = [g for g in ard.getHeaders()["guids"] if g not in ingroup_guids]

    allele_ids = [x[0] for x in model]
    im, ih = ard.subset(allele_list=allele_ids, guid_list=ingroup_guids)
    om, oh = ard.subset(allele_list=allele_ids, guid_list=outgroup_guids)

    af_distributions = freqs([im, om])

    print(af_distributions)

    mat, headers = ard.subset(allele_list=allele_ids)

    for i, q in enumerate(mat):
        guid = headers['guids'][i]
        guid_pop = meta[meta['guid']==guid]['subtype'].values[0]

        log_likelihoods = compute_likelihoods(q, af_distributions)
        probs = compute_bayes(log_likelihoods)

        r2[guid]["pop"] = guid_pop
        r2[guid][pop] = probs[0]

        # if guid in ingroup_guids and probs[0]>=t:
        #     results["TP"] += 1

        # if guid in ingroup_guids and probs[0]<t:
        #     results["FN"] += 1

        # if guid in outgroup_guids and probs[0]>=t:
        #     results["FP"] += 1
        
        # if guid in outgroup_guids and probs[0]<t:
        #     results["TN"] += 1

Running model: AF-NE
[array([0.56818182, 0.56818182, 0.70454545, 0.63636364, 0.54545455,
       0.56818182, 0.54545455, 0.54545455, 0.54545455, 0.52272727,
       0.54545455, 0.52272727, 0.68181818, 0.54545455, 0.54545455,
       0.54545455, 0.54545455, 0.5       , 0.52272727, 0.40909091,
       0.84090909, 0.40909091, 0.95454545, 0.97727273, 0.81818182,
       0.70454545, 0.47727273, 0.47727273, 0.65909091, 0.54545455,
       0.34090909, 0.97727273, 0.97727273, 0.72727273, 0.63636364,
       0.72727273, 0.38636364, 0.95454545, 0.52272727, 0.61363636,
       0.52272727, 0.34090909]), array([0.00190404, 0.00304646, 0.0156131 , 0.00799695, 0.00228484,
       0.00418888, 0.00304646, 0.00342727, 0.00342727, 0.00266565,
       0.0053313 , 0.00380807, 0.02361005, 0.00685453, 0.00685453,
       0.00799695, 0.00837776, 0.00685453, 0.01142422, 0.00228484,
       0.11309977, 0.00342727, 0.20944402, 0.24409749, 0.10967251,
       0.06169078, 0.0106626 , 0.01180503, 0.05178979, 0.02437167,
       

In [38]:
pd.DataFrame(r2).T

,pop,AS-SE-E
FP0028-C,AF-W,0.0
FP0045-CW,AF-W,0.0
FP0046-C,AF-W,0.0
FP0050-CW,AF-W,0.0
FP0095-C,AF-W,0.0
...,...,...
SPT43364,AF-E,0.0
SPT43371,AF-E,0.0
SPT43373,AF-E,0.0
SPT43376,AF-E,0.0
